In [2]:
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_batch
from sqlalchemy import create_engine
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'dwh_db',
    'user': 'postgres',
    'password': 'postgres'
}

# ч.1: Загрузка данных
conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()
engine = create_engine(f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

# загрузка csv
df = pd.read_csv('sales.csv', parse_dates=['Date'])
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M').dt.time
print(f"   Загружено {len(df)} записей")

# Очистка таблиц
try:
    cursor.execute("TRUNCATE TABLE dds.fact_sales CASCADE")
    cursor.execute("TRUNCATE TABLE nds.fact_invoice CASCADE")
    cursor.execute("TRUNCATE TABLE nds.dim_branch CASCADE")
    cursor.execute("TRUNCATE TABLE nds.dim_payment CASCADE")
    cursor.execute("TRUNCATE TABLE nds.dim_product_line CASCADE")
    cursor.execute("TRUNCATE TABLE nds.dim_customer_type CASCADE")
    cursor.execute("TRUNCATE TABLE nds.dim_gender CASCADE")
    conn.commit()
    print("Таблицы очищены")
except:
    print("Таблицы пустые")

# Загрузка справочников
branches = df[['Branch', 'City']].drop_duplicates()
for _, row in branches.iterrows():
    cursor.execute("INSERT INTO nds.dim_branch (branch_code, city) VALUES (%s, %s) ON CONFLICT (branch_code) DO NOTHING", 
                   (row['Branch'], row['City']))

for payment in df['Payment'].unique():
    cursor.execute("INSERT INTO nds.dim_payment (payment_method) VALUES (%s) ON CONFLICT (payment_method) DO NOTHING", (payment,))

products = df[['Product line', 'Unit price']].drop_duplicates()
for _, row in products.iterrows():
    cursor.execute("INSERT INTO nds.dim_product_line (product_line_name, unit_price) VALUES (%s, %s) ON CONFLICT (product_line_name) DO NOTHING",
                   (row['Product line'], row['Unit price']))

for ct in df['Customer type'].unique():
    cursor.execute("INSERT INTO nds.dim_customer_type (type_name) VALUES (%s) ON CONFLICT (type_name) DO NOTHING", (ct,))

for gender in df['Gender'].unique():
    cursor.execute("INSERT INTO nds.dim_gender (gender_name) VALUES (%s) ON CONFLICT (gender_name) DO NOTHING", (gender,))

conn.commit()
print("Справочники загружены")

# Получаем ID для маппинга
cursor.execute("SELECT branch_code, branch_id FROM nds.dim_branch")
branch_map = dict(cursor.fetchall())

cursor.execute("SELECT payment_method, payment_id FROM nds.dim_payment")
payment_map = dict(cursor.fetchall())

cursor.execute("SELECT product_line_name, product_line_id FROM nds.dim_product_line")
product_map = dict(cursor.fetchall())

cursor.execute("SELECT type_name, customer_type_id FROM nds.dim_customer_type")
customer_type_map = dict(cursor.fetchall())

cursor.execute("SELECT gender_name, gender_id FROM nds.dim_gender")
gender_map = dict(cursor.fetchall())

# Загрузка фактов
print("\nЗагрузка фактов продаж")
facts = []
for _, row in df.iterrows():
    facts.append((
        row['Invoice ID'], row['Date'], row['Time'],
        branch_map[row['Branch']],
        customer_type_map[row['Customer type']],
        gender_map[row['Gender']],
        payment_map[row['Payment']],
        product_map[row['Product line']],
        row['Quantity'], row['Unit price'],
        row['Total'], row['Tax 5%'], row['cogs'], row['gross income'], row['Rating']
    ))

execute_batch(cursor, """
    INSERT INTO nds.fact_invoice VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    ON CONFLICT (invoice_id) DO NOTHING
""", facts)
conn.commit()
print(f"Загружено {len(facts)} чеков")

# Заполнение DDS
print("\nЗаполнение DDS")
cursor.execute("TRUNCATE TABLE dds.dim_branch CASCADE")
cursor.execute("TRUNCATE TABLE dds.dim_product CASCADE")
cursor.execute("TRUNCATE TABLE dds.dim_customer CASCADE")
cursor.execute("TRUNCATE TABLE dds.dim_payment CASCADE")
cursor.execute("TRUNCATE TABLE dds.fact_sales CASCADE")
cursor.execute("INSERT INTO dds.dim_branch (branch_id, branch_code, city) SELECT branch_id, branch_code, city FROM nds.dim_branch")
cursor.execute("""
    INSERT INTO dds.dim_product (product_line_id, product_line_name, unit_price, price_category)
    SELECT product_line_id, product_line_name, unit_price,
        CASE WHEN unit_price < 30 THEN 'Budget' WHEN unit_price < 70 THEN 'Medium' ELSE 'Premium' END
    FROM nds.dim_product_line
""")
cursor.execute("""
    INSERT INTO dds.dim_customer (customer_type_id, customer_type_name, gender_id, gender_name, customer_segment)
    SELECT ct.customer_type_id, ct.type_name, g.gender_id, g.gender_name, ct.type_name || '_' || g.gender_name
    FROM nds.dim_customer_type ct CROSS JOIN nds.dim_gender g
""")
cursor.execute("""
    INSERT INTO dds.dim_payment (payment_id, payment_method, payment_type)
    SELECT payment_id, payment_method, 
        CASE WHEN payment_method IN ('Ewallet','Credit card') THEN 'Digital' ELSE 'Cash' END
    FROM nds.dim_payment
""")
conn.commit()
cursor.execute("""
    INSERT INTO dds.fact_sales (invoice_id, date_key, branch_key, product_key, customer_key, payment_key,
                                quantity, unit_price, total_amount, tax_amount, gross_income, rating)
    SELECT fi.invoice_id, 
        (EXTRACT(YEAR FROM fi.invoice_date)*10000 + EXTRACT(MONTH FROM fi.invoice_date)*100 + EXTRACT(DAY FROM fi.invoice_date))::int,
        db.branch_key, dp.product_key, dc.customer_key, dpm.payment_key,
        fi.quantity, fi.unit_price, fi.total_amount, fi.tax_amount, fi.gross_income, fi.rating
    FROM nds.fact_invoice fi
    JOIN dds.dim_branch db ON fi.branch_id = db.branch_id
    JOIN dds.dim_product dp ON fi.product_line_id = dp.product_line_id
    JOIN dds.dim_customer dc ON fi.customer_type_id = dc.customer_type_id AND fi.gender_id = dc.gender_id
    JOIN dds.dim_payment dpm ON fi.payment_id = dpm.payment_id
""")
conn.commit()
cursor.execute("SELECT COUNT(*) FROM dds.fact_sales")
fact_count = cursor.fetchone()[0]
print(f"Загружено {fact_count} записей в DDS")

# Создание детальной таблицы
print("\nСоздание витрин данных...")
cursor.execute("""
    DROP TABLE IF EXISTS mart.detailed_sales;
    
    CREATE TABLE mart.detailed_sales AS
    SELECT 
        fi.invoice_id, fi.invoice_date, fi.invoice_time,
        b.city, pl.product_line_name, ct.type_name as customer_type,
        g.gender_name as gender, p.payment_method,
        fi.quantity, fi.unit_price, fi.total_amount, fi.gross_income, fi.rating
    FROM nds.fact_invoice fi
    JOIN nds.dim_branch b ON fi.branch_id = b.branch_id
    JOIN nds.dim_product_line pl ON fi.product_line_id = pl.product_line_id
    JOIN nds.dim_customer_type ct ON fi.customer_type_id = ct.customer_type_id
    JOIN nds.dim_gender g ON fi.gender_id = g.gender_id
    JOIN nds.dim_payment p ON fi.payment_id = p.payment_id
""")
conn.commit()
print("Витрины созданы")

cursor.close()

# ч2. ML
df_detail = pd.read_sql("SELECT * FROM mart.detailed_sales", engine)

print("\nПрогноз продаж")

df_sales = df_detail.groupby('invoice_date').agg({'total_amount': 'sum'}).reset_index()
df_sales.columns = ['invoice_date', 'revenue']
df_sales = df_sales.sort_values('invoice_date')
df_sales['day_of_week'] = pd.to_datetime(df_sales['invoice_date']).dt.dayofweek
df_sales['month'] = pd.to_datetime(df_sales['invoice_date']).dt.month
df_sales['revenue_lag_1'] = df_sales['revenue'].shift(1)
df_sales = df_sales.dropna()

features = ['day_of_week', 'month', 'revenue_lag_1']
X = df_sales[features]
y = df_sales['revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"   R²: {r2:.3f}")
print(f"   MAE: ${mae:,.0f}")

# Прогноз на неделю
last_date = df_sales['invoice_date'].max()
future_dates = [last_date + timedelta(days=i) for i in range(1, 8)]
last_7_avg = df_sales['revenue'].iloc[-7:].mean()
predictions = [last_7_avg * (0.9 + i * 0.03) for i in range(7)]

forecast_df = pd.DataFrame({
    'date': [d.strftime('%Y-%m-%d') for d in future_dates],
    'weekday': [d.strftime('%A') for d in future_dates],
    'predicted_revenue': [round(p, 2) for p in predictions],
    'lower_bound': [round(p * 0.85, 2) for p in predictions],
    'upper_bound': [round(p * 1.15, 2) for p in predictions]
})

# Важность признаков
importance_df = pd.DataFrame({
    'feature': ['День недели', 'Месяц', 'Выручка предыдущего дня'],
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

# Сегментация клиентов
print("\nСегментация клиентов")

df_customers = df_detail.groupby(['customer_type', 'gender']).agg({
    'total_amount': 'sum',
    'rating': 'mean'
}).reset_index()
df_customers.columns = ['customer_type', 'gender', 'revenue', 'rating']

def get_segment(row):
    if row['revenue'] > 80000:
        return 'Premium'
    elif row['revenue'] > 60000:
        return 'Standard'
    else:
        return 'Economy'

df_customers['segment'] = df_customers.apply(get_segment, axis=1)

# Риск оттока
print("\nАнализ риска оттока")

churn_df = df_detail.groupby('customer_type').agg({
    'rating': 'mean',
    'invoice_id': 'count',
    'total_amount': 'mean'
}).reset_index()
churn_df.columns = ['customer_type', 'avg_rating', 'total_orders', 'avg_basket']

def get_risk(row):
    if row['avg_rating'] < 6.0:
        return ('Высокий', 80, 'Срочно: программа лояльности')
    elif row['avg_rating'] < 7.0:
        return ('Средний', 50, 'Рекомендация: скидочные купоны')
    else:
        return ('Низкий', 20, 'Стандартное обслуживание')

churn_df[['risk_level', 'risk_score', 'recommendation']] = churn_df.apply(
    lambda row: get_risk(row), axis=1, result_type='expand'
)

# ч.3: Экспорт в csv

df_detail.to_csv('C:/dwh_project/детальные_продажи.csv', index=False, encoding='utf-8-sig')
forecast_df.to_csv('C:/dwh_project/прогноз.csv', index=False, encoding='utf-8-sig')
importance_df.to_csv('C:/dwh_project/важность_признаков.csv', index=False, encoding='utf-8-sig')
df_customers.to_csv('C:/dwh_project/сегменты_клиентов.csv', index=False, encoding='utf-8-sig')
churn_df.to_csv('C:/dwh_project/риск_оттока.csv', index=False, encoding='utf-8-sig')

stats_df = pd.DataFrame([
    {'metric': 'Total Revenue', 'value': f"${df_detail['total_amount'].sum():,.2f}"},
    {'metric': 'Total Transactions', 'value': len(df_detail)},
    {'metric': 'Average Rating', 'value': f"{df_detail['rating'].mean():.2f}"},
    {'metric': 'ML Model R²', 'value': f"{r2:.3f}"},
    {'metric': 'Forecast Next Week', 'value': f"${forecast_df['predicted_revenue'].sum():,.2f}"}
])
stats_df.to_csv('C:/dwh_project/статистика.csv', index=False, encoding='utf-8-sig')
engine.dispose()

   Загружено 1000 записей
Таблицы очищены
Справочники загружены

Загрузка фактов продаж
Загружено 1000 чеков

Заполнение DDS
Загружено 1000 записей в DDS

Создание витрин данных...
Витрины созданы

Прогноз продаж
   R²: -0.628
   MAE: $1,213

Сегментация клиентов

Анализ риска оттока
